<a href="https://colab.research.google.com/github/DtDOW/nnU-Net-using-custom-loss-function-for-Prostate-Cancer-Lesion-Segmentation-in-PSMA-PET-CT/blob/main/nnUNetv2_PETCT_CustomLoss_51Epoch_100Cases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# nnU-Net v2 PET/CT — CUSTOM LOSS | **51 epochs × 100 studies**

**Goal:** Train **from scratch** with the original nnU-Net 3D full-resolution plans on **exactly 100** previously preprocessed paired PET/CT studies (`CT _0000`, `PET _0001`, binary lesion mask). Uses fold `0`, seed `42`, and the original training schedule for **51 epochs**.

- **Trainer:** `nnUNetTrainerCoverageAlign`. Original loss preserved *exactly* from the uploaded old custom notebook: default nnU-Net Dice+CE + missed-lesion-area (0.50) + centroid-alignment (0.25) + orientation (0.10), grid size 32.
- **Inputs:** `/content/drive/MyDrive/nnUNet_shared/nnUNet_preprocessed/Dataset901_PETCT_Baseline`.
- **Critical safety check:** if the cache contains **anything other than 100 fully preprocessed studies**, the notebook **stops before training**. It never silently falls back to the old nine-study dataset. It does **not** rerun preprocessing.
- **Output:** new unique local and Drive folders; no old or opposite-model checkpoints are overwritten. It records a manifest including all case IDs, fold IDs, original loss parameters, and checkpoint trainer names.
- `USE_LOCAL_PREPROCESSED_COPY = False` is fine: source arrays are read directly from Drive and **training still runs on the GPU**. Model checkpoints are produced locally, then backed up to Drive after training.

**Run:** Colab → Runtime → Change runtime type → GPU → Run all. The final saving cell must complete before the runtime disconnects. Training takes 51 nnU-Net epochs (typically 250 optimizer steps per epoch, not 51 passes over the data).

## 0. Configuration — intentionally strict and isolated

In [ ]:
# === DATA — same existing preprocessed dataset for both experiments ===
DRIVE_SHARED_ROOT = "/content/drive/MyDrive/nnUNet_shared"
DATASET_ID = 901
DATASET_NAME = "PETCT_Baseline"
MODEL_CONFIGURATION = "3d_fullres"
FOLD = 0
EXPECTED_NUM_CASES = 84   # STOP if not exactly 100 preprocessed PET/CT studies

# === TRAINING ===
EPOCHS = 51
DEVICE = "auto"               # cuda if available
RANDOM_SEED = 42
USE_LOCAL_PREPROCESSED_COPY = False  # False: read cache from Drive; True: copy cache locally first
LOCAL_WORK_DIR = "/content/nnUNet_PETCT_CustomLoss_51epochs_100cases"

# === OUTPUT — intentionally NOT the old baseline_results folder ===
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/nnUNet_PETCT_CustomLoss_51epochs_100cases"
EXPERIMENT_LABEL = "CustomLoss_51epochs_100cases_fold0"
SAVE_RESULTS_TO_DRIVE = True
TRAINER_NAME = "nnUNetTrainerCoverageAlign"
IS_CUSTOM_LOSS = True

# === ORIGINAL CUSTOM LOSS (identical settings as earlier experiment) ===
W_MISSED_AREA = 0.50
W_ALIGNMENT = 0.25
W_ORIENTATION = 0.10
ALIGN_GRID_MAX_SIZE = 32

# Guard: do not silently reuse/overwrite a prior experiment.
ALLOW_OVERWRITE = False  # Leave False for a fair, fresh training run

## 1. Install dependencies and mount Google Drive

In [ ]:
%pip -q install nnunetv2 SimpleITK tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Verify that all **100** preprocessed studies are present (no new preprocessing)

In [ ]:
import os, json, random, hashlib, shutil, time
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import torch

assert EPOCHS == 51, "This notebook is deliberately configured for 51 epochs."
assert EXPECTED_NUM_CASES == 84, "This experiment requires exactly 100 studies."
assert MODEL_CONFIGURATION in ("2d", "3d_fullres")
assert all(w >= 0 for w in (W_MISSED_AREA, W_ALIGNMENT, W_ORIENTATION))
assert ALIGN_GRID_MAX_SIZE >= 4

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

if DEVICE == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else (
        "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"
    )
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. In Colab select Runtime > Change runtime type > GPU.")

shared_root = Path(DRIVE_SHARED_ROOT)
dataset_folder_name = f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"
shared_preproc_root = shared_root / "nnUNet_preprocessed"
shared_ds = shared_preproc_root / dataset_folder_name
shared_raw_root = shared_root / "nnUNet_raw"
work_dir = Path(LOCAL_WORK_DIR)
work_dir.mkdir(parents=True, exist_ok=True)
run_info_dir = work_dir / "run_info"
run_info_dir.mkdir(parents=True, exist_ok=True)

if not shared_ds.is_dir():
    raise FileNotFoundError(
        f"Preprocessed dataset not found: {shared_ds}\n"
        "Save preprocessing to Drive first; do not rerun preprocessing just for this notebook."
    )

plans_path = shared_ds / "nnUNetPlans.json"
dataset_path = shared_ds / "dataset.json"
for f in (plans_path, dataset_path):
    if not f.is_file():
        raise FileNotFoundError(f"Missing required preprocessing metadata: {f}")
plans = json.loads(plans_path.read_text())
dataset_info = json.loads(dataset_path.read_text())
configs = plans.get("configurations", {})
if MODEL_CONFIGURATION not in configs:
    raise ValueError(f"Configuration {MODEL_CONFIGURATION!r} not found in nnUNetPlans.json. Available: {list(configs)}")

preprocessed_identifier = configs[MODEL_CONFIGURATION].get(
    "data_identifier", f"nnUNetPlans_{MODEL_CONFIGURATION}"
)
shared_case_dir = shared_ds / preprocessed_identifier
if not shared_case_dir.is_dir():
    raise FileNotFoundError(f"Missing preprocessed case directory: {shared_case_dir}")

expected_case_count = int(dataset_info.get("numTraining", 0))
preprocessed_case_ids = sorted(p.stem for p in shared_case_dir.glob("*.pkl"))
if not preprocessed_case_ids:
    raise RuntimeError(f"No preprocessed .pkl case metadata found in {shared_case_dir}")
if len(preprocessed_case_ids) != expected_case_count:
    raise RuntimeError(
        f"Preprocessed files/metadata mismatch: {len(preprocessed_case_ids)} case files "
        f"versus dataset.json numTraining={expected_case_count}. Check cache completeness."
    )
if EXPECTED_NUM_CASES is not None and len(preprocessed_case_ids) != EXPECTED_NUM_CASES:
    raise ValueError(
        f"You expected {EXPECTED_NUM_CASES} cases but this cache has {len(preprocessed_case_ids)}. "
        "To train on a different subset, create a different preprocessed dataset ID."
    )

channels = dataset_info.get("channel_names", {})
labels = dataset_info.get("labels", {})
if len(channels) != 2 or labels.get("background") != 0 or labels.get("lesion") != 1:
    raise ValueError(f"Expected CT + PET channels and binary lesion labels, got: {channels}, {labels}")

# Modern nnU-Net stores .b2nd and _seg.b2nd; some versions use .npz.
def cached_case_ok(case):
    p = shared_case_dir
    modern = (p / f"{case}.b2nd").is_file() and (p / f"{case}_seg.b2nd").is_file()
    legacy = (p / f"{case}.npz").is_file()
    return (p / f"{case}.pkl").is_file() and (modern or legacy)

missing_cases = [case for case in preprocessed_case_ids if not cached_case_ok(case)]
if missing_cases:
    raise RuntimeError(f"{len(missing_cases)} preprocessed cases are incomplete, e.g.: {missing_cases[:3]}")

# Avoid silently reusing a split created for a different dataset subset.
existing_split_file = shared_ds / "splits_final.json"
if existing_split_file.is_file():
    split_data = json.loads(existing_split_file.read_text())
    if FOLD >= len(split_data):
        raise ValueError(f"Fold {FOLD} absent in {existing_split_file}")
    split = split_data[FOLD]
    train_set, val_set = set(split["train"]), set(split["val"])
    actual_case_set = set(preprocessed_case_ids)
    if train_set & val_set or train_set | val_set != actual_case_set:
        raise RuntimeError(
            "Saved splits_final.json does not match this preprocessed dataset. "
            "Do not compare models until the split is corrected. "
            f"Fold train={len(train_set)}, val={len(val_set)}, dataset={len(actual_case_set)}"
        )
    print(f"Fold {FOLD}: {len(train_set)} train, {len(val_set)} validation cases")
else:
    print("No saved splits_final.json yet; nnU-Net will generate a deterministic split during training.")

print("Dataset:", dataset_folder_name)
print("Preprocessing:", shared_case_dir)
print("Available cases (actual):", len(preprocessed_case_ids))
print("Configuration:", MODEL_CONFIGURATION, "| Fold:", FOLD)
print("Epochs:", EPOCHS, "| Device:", DEVICE)
print("Case ID checksum:", hashlib.sha256("\n".join(preprocessed_case_ids).encode()).hexdigest()[:16])
print("No images are revalidated or preprocessed here.")

# Explicitly print the fold source; the baseline and custom must use the same file.
if existing_split_file.exists():
    print("Verified fold file:", existing_split_file)
else:
    print("No fold file yet. nnU-Net will generate its standard 5-fold split during first training.")

# This dataset ID is a name, NOT evidence that the dataset is still the original 9-case subset.
print("IMPORTANT: this dataset contains exactly", len(preprocessed_case_ids), "paired CT/PET studies.")

Fold 0: 67 train, 17 validation cases
Dataset: Dataset901_PETCT_Baseline
Preprocessing: /content/drive/MyDrive/nnUNet_shared/nnUNet_preprocessed/Dataset901_PETCT_Baseline/nnUNetPlans_3d_fullres
Available cases (actual): 84
Configuration: 3d_fullres | Fold: 0
Epochs: 51 | Device: cuda
Case ID checksum: bef90b1633bfa8c5
No images are revalidated or preprocessed here.
Verified fold file: /content/drive/MyDrive/nnUNet_shared/nnUNet_preprocessed/Dataset901_PETCT_Baseline/splits_final.json
IMPORTANT: this dataset contains exactly 84 paired CT/PET studies.


## 3. Set nnU-Net data and result paths (optionally copy preprocessed arrays to local storage)

In [ ]:
active_preproc_root = shared_preproc_root
local_preproc_root = work_dir / "nnUNet_preprocessed"
local_ds = local_preproc_root / dataset_folder_name

if USE_LOCAL_PREPROCESSED_COPY:
    # Copy dataset-level files and the selected configuration (plus validation GT masks).
    to_copy = [p for p in shared_ds.iterdir() if p.is_file()]
    for subdir in (shared_ds / preprocessed_identifier, shared_ds / "gt_segmentations"):
        if subdir.is_dir():
            to_copy.extend(p for p in subdir.rglob("*") if p.is_file())

    def needs_copy(source):
        destination = local_ds / source.relative_to(shared_ds)
        return not (destination.is_file() and destination.stat().st_size == source.stat().st_size)

    remaining = [p for p in to_copy if needs_copy(p)]
    remaining_bytes = sum(p.stat().st_size for p in remaining)
    free_bytes = shutil.disk_usage(work_dir).free
    print(f"Local copy required: {remaining_bytes / 2**30:.2f} GiB; free: {free_bytes / 2**30:.2f} GiB")

    if remaining_bytes * 1.15 > free_bytes:
        print("Not enough local disk. Training directly from the shared Drive cache instead.")
    else:
        for src_file in tqdm(remaining, desc="Copying preprocessed cache", unit="file"):
            dest_file = local_ds / src_file.relative_to(shared_ds)
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            # Use a temporary file to avoid leaving a complete-looking partial file.
            temp_file = dest_file.with_name(dest_file.name + ".partial")
            try:
                shutil.copy2(src_file, temp_file)
                temp_file.replace(dest_file)
            finally:
                temp_file.unlink(missing_ok=True)
        active_preproc_root = local_preproc_root
        print("Using local preprocessed cache:", local_ds)
else:
    print("Using preprocessed cache on Drive.")

# Set ONCE before importing nnU-Net training machinery.
results_root = work_dir / "nnUNet_results"
results_root.mkdir(parents=True, exist_ok=True)
raw_root_for_env = shared_raw_root if shared_raw_root.is_dir() else (work_dir / "nnUNet_raw")
raw_root_for_env.mkdir(parents=True, exist_ok=True)
os.environ["nnUNet_raw"] = str(raw_root_for_env)
os.environ["nnUNet_preprocessed"] = str(active_preproc_root)
os.environ["nnUNet_results"] = str(results_root)

active_ds = active_preproc_root / dataset_folder_name
assert (active_ds / "nnUNetPlans.json").is_file()
assert (active_ds / preprocessed_identifier).is_dir()
print("nnUNet_preprocessed =", os.environ["nnUNet_preprocessed"])
print("nnUNet_results =", os.environ["nnUNet_results"])

Using preprocessed cache on Drive.
nnUNet_preprocessed = /content/drive/MyDrive/nnUNet_shared/nnUNet_preprocessed
nnUNet_results = /content/nnUNet_PETCT_CustomLoss_51epochs_100cases/nnUNet_results


## 4. Check checkpoint destination **before training**

In [ ]:
from pathlib import Path

# The dedicated output paths ensure a baseline can never accidentally be saved as custom, or vice versa.
expected_local_fold = (
    results_root / dataset_folder_name
    / f"{TRAINER_NAME}__nnUNetPlans__{MODEL_CONFIGURATION}"
    / f"fold_{FOLD}"
)
experiment_drive_root = Path(DRIVE_RESULTS_ROOT) / dataset_folder_name
expected_drive_legacy = experiment_drive_root / "trained_model"

if not ALLOW_OVERWRITE:
    local_old = list(expected_local_fold.glob("checkpoint_*.pth")) if expected_local_fold.is_dir() else []
    drive_old = list(expected_drive_legacy.glob("checkpoint_*.pth")) if expected_drive_legacy.is_dir() else []
    if local_old or drive_old:
        raise RuntimeError(
            "Checkpoint files already exist in this isolated experiment. "
            "To protect them, choose new LOCAL_WORK_DIR and DRIVE_RESULTS_ROOT paths "
            "rather than rerunning in the same folders. "
            f"Local={local_old}, Drive={drive_old}"
        )

print("Trainer:", TRAINER_NAME)
print("Local checkpoint directory:", expected_local_fold)
print("Drive backup directory:", expected_drive_legacy)
print("Preprocessed case count:", len(preprocessed_case_ids))
assert len(preprocessed_case_ids) == 84
assert TRAINER_NAME == ("nnUNetTrainerCoverageAlign" if IS_CUSTOM_LOSS else "nnUNetTrainer")

Trainer: nnUNetTrainerCoverageAlign
Local checkpoint directory: /content/nnUNet_PETCT_CustomLoss_51epochs_100cases/nnUNet_results/Dataset901_PETCT_Baseline/nnUNetTrainerCoverageAlign__nnUNetPlans__3d_fullres/fold_0
Drive backup directory: /content/drive/MyDrive/nnUNet_PETCT_CustomLoss_51epochs_100cases/Dataset901_PETCT_Baseline/trained_model
Preprocessed case count: 84


## 5. Install your **original unchanged** custom-loss trainer

In [ ]:
import inspect, importlib
import nnunetv2

for name, val in {
    "NNUNET_CUSTOM_W_MISSED": W_MISSED_AREA,
    "NNUNET_CUSTOM_W_ALIGN": W_ALIGNMENT,
    "NNUNET_CUSTOM_W_ORIENT": W_ORIENTATION,
    "NNUNET_CUSTOM_ALIGN_GRID": ALIGN_GRID_MAX_SIZE,
}.items():
    os.environ[name] = str(val)

custom_trainer_path = (
    Path(inspect.getfile(nnunetv2)).resolve().parent
    / "training" / "nnUNetTrainer" / "variants" / "loss"
    / "nnUNetTrainerCoverageAlign.py"
)
custom_trainer_path.parent.mkdir(parents=True, exist_ok=True)

TRAINER_SOURCE = '''# This file is generated by the PET/CT custom-loss notebook.
import os
import torch
from torch import nn
from torch.nn import functional as F
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class CoverageAlignmentLoss(nn.Module):
    """Adds differentiable binary-lesion penalties to nnU-Net's own loss."""

    def __init__(self, base_loss, w_missed=0.5, w_align=0.25, w_orient=0.1,
                 align_grid=32, ignore_label=None, eps=1e-6):
        super().__init__()
        self.base_loss = base_loss
        self.w_missed = float(w_missed)
        self.w_align = float(w_align)
        self.w_orient = float(w_orient)
        self.align_grid = int(align_grid)
        self.ignore_label = ignore_label
        self.eps = float(eps)
        if self.align_grid < 4:
            raise ValueError("align_grid must be >= 4")

    def _active_mean(self, per_case, active):
        active = active.to(per_case.dtype)
        return (per_case * active).sum() / active.sum().clamp_min(1.0)

    def _centroid_and_moments(self, weights):
        # weights: [B,1,H,W] or [B,1,D,H,W]
        dims = weights.shape[2:]
        axes = [torch.linspace(-1, 1, int(d), device=weights.device, dtype=weights.dtype)
                for d in dims]
        meshes = torch.meshgrid(*axes, indexing="ij")
        coords = torch.stack(meshes, dim=-1).reshape(-1, len(dims))
        w = weights.reshape(weights.shape[0], -1)
        mass = w.sum(dim=1).clamp_min(self.eps)
        mu = (w @ coords) / mass.unsqueeze(1)
        xc = coords.unsqueeze(0) - mu.unsqueeze(1)
        cov = torch.einsum("bn,bni,bnj->bij", w, xc, xc) / mass[:, None, None]
        trace = cov.diagonal(dim1=1, dim2=2).sum(dim=1).clamp_min(self.eps)
        cov_normalized = cov / trace[:, None, None]
        return mu, cov_normalized

    def forward(self, net_output, target):
        # Preserve nnU-Net's original Dice+CE / deep-supervision implementation.
        base = self.base_loss(net_output, target)
        if self.w_missed == 0 and self.w_align == 0 and self.w_orient == 0:
            return base

        logits = net_output[0] if isinstance(net_output, (list, tuple)) else net_output
        truth = target[0] if isinstance(target, (list, tuple)) else target
        if logits.ndim not in (4, 5) or logits.shape[1] != 2 or truth.shape[1] != 1:
            raise ValueError("This custom loss requires 2D/3D binary foreground labels, 2 output logits")

        # float32 for stable, differentiable probability / spatial calculations.
        prob = torch.softmax(logits.float(), dim=1)[:, 1:2]
        gt = (truth == 1).to(prob.dtype)
        if self.ignore_label is not None:
            valid = (truth != self.ignore_label).to(prob.dtype)
            prob = prob * valid
            gt = gt * valid

        sums = tuple(range(1, prob.ndim))
        gt_mass = gt.sum(dim=sums)
        active = gt_mass > 0   # no extra penalty on lesion-free samples
        aux = prob.new_zeros(())

        if self.w_missed:
            missed = (gt * (1.0 - prob)).sum(dim=sums) / gt_mass.clamp_min(self.eps)
            aux = aux + self.w_missed * self._active_mean(missed, active)

        if self.w_align or self.w_orient:
            shape = tuple(min(self.align_grid, int(s)) for s in prob.shape[2:])
            pool = F.adaptive_avg_pool2d if prob.ndim == 4 else F.adaptive_avg_pool3d
            p_small = pool(prob, shape)
            g_small = pool(gt, shape)
            p_mu, p_cov = self._centroid_and_moments(p_small)
            g_mu, g_cov = self._centroid_and_moments(g_small)

            if self.w_align:
                spatial_dims = logits.ndim - 2
                identity = torch.eye(spatial_dims, device=prob.device, dtype=prob.dtype)
                identity = identity.unsqueeze(0).expand(prob.shape[0], -1, -1)
                # Grid axes are (x,y,z); tensor axes are (z,y,x).
                delta = (g_mu - p_mu).flip(dims=[1])
                theta = torch.cat((identity, -delta.unsqueeze(-1)), dim=-1)
                sampling_grid = F.affine_grid(theta, p_small.size(), align_corners=True)
                translated = F.grid_sample(p_small, sampling_grid,
                                           mode="bilinear", padding_mode="zeros",
                                           align_corners=True)
                intersection = (translated * g_small).sum(dim=sums)
                denominator = translated.sum(dim=sums) + g_small.sum(dim=sums)
                aligned_dice = (2.0 * intersection + self.eps) / (denominator + self.eps)
                aux = aux + self.w_align * self._active_mean(1.0 - aligned_dice, active)

            if self.w_orient:
                # Second-moment matrix distance, not an arccos of Dice.
                # Orientation/shape of near-spherical objects is inherently ambiguous.
                moment_distance = (p_cov - g_cov).square().sum(dim=(1, 2))
                aux = aux + self.w_orient * self._active_mean(moment_distance, active)

        return base + aux


class nnUNetTrainerCoverageAlign(nnUNetTrainer):
    """Same nnU-Net architecture, augmentations and loss, with extra lesion penalties."""

    def _build_loss(self):
        if self.label_manager.has_regions:
            raise NotImplementedError("This notebook's custom loss expects non-region binary labels")
        lesion = self.dataset_json.get("labels", {}).get("lesion")
        if lesion != 1:
            raise ValueError("Expected dataset.json labels={'background': 0, 'lesion': 1}")
        unchanged_default_loss = super()._build_loss()
        return CoverageAlignmentLoss(
            unchanged_default_loss,
            w_missed=float(os.environ.get("NNUNET_CUSTOM_W_MISSED", "0.5")),
            w_align=float(os.environ.get("NNUNET_CUSTOM_W_ALIGN", "0.25")),
            w_orient=float(os.environ.get("NNUNET_CUSTOM_W_ORIENT", "0.1")),
            align_grid=int(os.environ.get("NNUNET_CUSTOM_ALIGN_GRID", "32")),
            ignore_label=self.label_manager.ignore_label,
        )
'''

custom_trainer_path.write_text(TRAINER_SOURCE, encoding="utf-8")
(run_info_dir / "nnUNetTrainerCoverageAlign.py").write_text(TRAINER_SOURCE, encoding="utf-8")
importlib.invalidate_caches()
from nnunetv2.training.nnUNetTrainer.variants.loss.nnUNetTrainerCoverageAlign import (
    nnUNetTrainerCoverageAlign, CoverageAlignmentLoss
)
print("Registered custom trainer:", custom_trainer_path)
print("Source saved:", run_info_dir / "nnUNetTrainerCoverageAlign.py")

Registered custom trainer: /usr/local/lib/python3.13/dist-packages/nnunetv2/training/nnUNetTrainer/variants/loss/nnUNetTrainerCoverageAlign.py
Source saved: /content/nnUNet_PETCT_CustomLoss_51epochs_100cases/run_info/nnUNetTrainerCoverageAlign.py


## 6. Test the custom loss on synthetic 2D and 3D patches (gradient and empty-mask check)

In [ ]:
import torch.nn.functional as F

class _DummyCrossEntropy(torch.nn.Module):
    def forward(self, logits, truth):
        return F.cross_entropy(logits, truth[:, 0].long())

for spatial in ((12, 16), (8, 10, 12)):
    logits = torch.randn((2, 2, *spatial), requires_grad=True)
    truth = torch.zeros((2, 1, *spatial), dtype=torch.long)
    if len(spatial) == 2:
        truth[0, 0, 3:8, 4:10] = 1
    else:
        truth[0, 0, 2:6, 3:8, 4:9] = 1

    loss = CoverageAlignmentLoss(
        _DummyCrossEntropy(), w_missed=W_MISSED_AREA,
        w_align=W_ALIGNMENT, w_orient=W_ORIENTATION,
        align_grid=ALIGN_GRID_MAX_SIZE
    )(logits, truth)
    loss.backward()
    assert torch.isfinite(loss).item() and torch.isfinite(logits.grad).all().item()
    assert logits.grad.abs().sum().item() > 0
    print(f"{len(spatial)}D loss/gradient check passed: loss={loss.item():.5f}")

# Confirm the exact training loss settings are active; no accidental all-zero weights.
assert W_MISSED_AREA == 0.50 and W_ALIGNMENT == 0.25 and W_ORIENTATION == 0.10
assert ALIGN_GRID_MAX_SIZE == 32
print("Custom loss weights verified.")

2D loss/gradient check passed: loss=1.38742
3D loss/gradient check passed: loss=1.36791
Custom loss weights verified.


## 7. Train **51 epochs** from randomly initialized weights

In [ ]:
from nnunetv2.run.run_training import get_trainer_from_args

# get_trainer_from_args does not load a checkpoint; this is a FRESH model.
trainer = get_trainer_from_args(
    dataset_name_or_id=str(DATASET_ID),
    configuration=MODEL_CONFIGURATION,
    fold=FOLD,
    trainer_name=TRAINER_NAME,
    device=torch.device(DEVICE),
)
trainer.num_epochs = EPOCHS

assert trainer.__class__.__name__ == TRAINER_NAME
assert trainer.num_epochs == 51
assert len(preprocessed_case_ids) == 84

print("TRAINING FROM SCRATCH")
print("Trainer:", trainer.__class__.__name__)
print("Epochs:", trainer.num_epochs)
print("Data:", len(preprocessed_case_ids), "paired CT/PET cases")
print("Fold:", FOLD)
print("Loss:", "nnU-Net Dice+CE + custom missed-area/alignment/orientation" if IS_CUSTOM_LOSS else "official nnU-Net Dice+CE")
print("Output:", trainer.output_folder)

trainer.run_training()
print("Training completed. Run the very next cell to back up the checkpoints to Drive.")

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

TRAINING FROM SCRATCH
Trainer: nnUNetTrainerCoverageAlign
Epochs: 51
Data: 84 paired CT/PET cases
Fold: 0
Loss: nnU-Net Dice+CE + custom missed-area/alignment/orientation
Output: /content/nnUNet_PETCT_CustomLoss_51epochs_100cases/nnUNet_results/Dataset901_PETCT_Baseline/nnUNetTrainerCoverageAlign__nnUNetPlans__3d_fullres/fold_0
2026-11-15 07:10:08.863032: Using torch.compile...
2026-11-15 07:10:12.217371: do_dummy_2d_data_aug: False
2026-11-15 07:10:12.219529: Using splits from existing split file: /content/drive/MyDrive/nnUNet_shared/nnUNet_preprocessed/Dataset901_PETC

W1115 07:11:40.008000 2515 torch/_inductor/utils.py:1806] [0/0] Not enough SMs to use max_autotune_gemm mode


2026-11-15 07:14:19.175689: train_loss 0.2132
2026-11-15 07:14:19.176444: val_loss 0.1705
2026-11-15 07:14:19.177133: Pseudo dice [np.float32(0.093)]
2026-11-15 07:14:19.177419: Epoch time: 174.41 s
2026-11-15 07:14:19.177688: Yayy! New best EMA pseudo Dice: 0.09300000220537186
2026-11-15 07:14:20.596338: 
2026-11-15 07:14:20.596861: Epoch 1
2026-11-15 07:14:20.597301: Current learning rate: 0.00982
2026-11-15 07:15:55.862081: train_loss 0.1398
2026-11-15 07:15:55.862532: val_loss 0.1661
2026-11-15 07:15:55.863039: Pseudo dice [np.float32(0.0359)]
2026-11-15 07:15:55.863225: Epoch time: 95.27 s
2026-11-15 07:15:57.210957: 
2026-11-15 07:15:57.211301: Epoch 2
2026-11-15 07:15:57.211836: Current learning rate: 0.00965
2026-11-15 07:17:32.295811: train_loss 0.0708
2026-11-15 07:17:32.296594: val_loss 0.1459
2026-11-15 07:17:32.296905: Pseudo dice [np.float32(0.0644)]
2026-11-15 07:17:32.297122: Epoch time: 95.09 s
2026-11-15 07:17:33.662069: 
2026-11-15 07:17:33.662581: Epoch 3
2026-11-15

KeyboardInterrupt: 

## 8. **Save and verify** the trained model, plans, fold, logs and run manifest

In [ ]:
import os, json, hashlib, shutil
from pathlib import Path
import torch

if not SAVE_RESULTS_TO_DRIVE:
    raise ValueError("SAVE_RESULTS_TO_DRIVE must be True to persist this trained model")

model_output = Path(trainer.output_folder)
if not model_output.is_dir():
    raise FileNotFoundError(f"Missing local model output: {model_output}")

checkpoint_files = sorted(p for p in model_output.glob("checkpoint_*.pth") if p.is_file())
final_checkpoint = model_output / "checkpoint_final.pth"
if not final_checkpoint.is_file():
    raise RuntimeError("Final checkpoint missing; training did not complete successfully. No final-model backup made.")

# Only load files created by this trusted notebook.
final_ckpt = torch.load(final_checkpoint, map_location="cpu", weights_only=False)
recorded_trainer = final_ckpt.get("trainer_name", "unknown")
recorded_epoch = final_ckpt.get("current_epoch", -1)
print("CHECKPOINT TRAINER:", recorded_trainer)
print("CHECKPOINT EPOCH:", recorded_epoch)
assert recorded_trainer == TRAINER_NAME, (
    f"WRONG TRAINER: expected {TRAINER_NAME}, checkpoint has {recorded_trainer}"
)
assert recorded_epoch == 51, (
    f"WRONG EPOCH COUNT: expected 51, checkpoint has {recorded_epoch}"
)
del final_ckpt

# Native nnU-Net inference folder (model + metadata + fold_0) plus a legacy flat copy
# compatible with your existing seven-case comparison notebook.
model_name = f"{TRAINER_NAME}__nnUNetPlans__{MODEL_CONFIGURATION}"
experiment_root = Path(DRIVE_RESULTS_ROOT) / dataset_folder_name
canonical_model_dir = experiment_root / "nnUNet_results" / dataset_folder_name / model_name
canonical_fold_dir = canonical_model_dir / f"fold_{FOLD}"
legacy_model_dir = experiment_root / "trained_model"
info_dir = experiment_root / "run_info" / EXPERIMENT_LABEL

for folder in (canonical_fold_dir, legacy_model_dir, info_dir):
    folder.mkdir(parents=True, exist_ok=True)

def safe_copy(src, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".partial")
    try:
        shutil.copy2(src, tmp)
        os.replace(tmp, dest)
    finally:
        tmp.unlink(missing_ok=True)

# Save checkpoint(s) AND trainer logs from the actual run.
for source in model_output.iterdir():
    if source.is_file():
        safe_copy(source, canonical_fold_dir / source.name)
for source in checkpoint_files:
    safe_copy(source, legacy_model_dir / source.name)

for dest_name, cache_name in (("plans.json", "nnUNetPlans.json"), ("dataset.json", "dataset.json")):
    source = model_output.parent / dest_name
    if not source.is_file():
        source = active_ds / cache_name
    if not source.is_file():
        raise FileNotFoundError(f"Missing model metadata: {source}")
    safe_copy(source, canonical_model_dir / dest_name)
    safe_copy(source, legacy_model_dir / dest_name)

# Capture the exact train/validation split generated or reused by nnU-Net.
source_split = active_ds / "splits_final.json"
if not source_split.is_file():
    raise FileNotFoundError(f"Training split is missing after training: {source_split}")
fold_data = json.loads(source_split.read_text())[FOLD]
train_ids, validation_ids = sorted(fold_data["train"]), sorted(fold_data["val"])
assert not (set(train_ids) & set(validation_ids))
assert set(train_ids) | set(validation_ids) == set(preprocessed_case_ids)
safe_copy(source_split, info_dir / "splits_final.json")

# If training was on local preprocessed copy, make sure baseline can reuse its split too.
shared_split = shared_ds / "splits_final.json"
if source_split.resolve() != shared_split.resolve():
    if shared_split.is_file():
        if json.loads(shared_split.read_text())[FOLD] != fold_data:
            raise RuntimeError("Local split differs from shared split: STOP, not a fair comparison")
    else:
        safe_copy(source_split, shared_split)

if IS_CUSTOM_LOSS:
    safe_copy(custom_trainer_path, info_dir / "nnUNetTrainerCoverageAlign.py")

manifest = {
    "experiment_label": EXPERIMENT_LABEL,
    "trainer_name": TRAINER_NAME,
    "epochs": EPOCHS,
    "configuration": MODEL_CONFIGURATION,
    "fold": FOLD,
    "data_source": str(shared_ds),
    "preprocessed_identifier": preprocessed_identifier,
    "case_count": len(preprocessed_case_ids),
    "case_ids": preprocessed_case_ids,
    "case_ids_sha256": hashlib.sha256("\n".join(preprocessed_case_ids).encode()).hexdigest(),
    "train_case_count": len(train_ids),
    "validation_case_count": len(validation_ids),
    "train_ids": train_ids,
    "validation_ids": validation_ids,
    "split_sha256": hashlib.sha256(json.dumps(fold_data, sort_keys=True).encode()).hexdigest(),
    "random_seed": RANDOM_SEED,
    "loss": "Default nnU-Net Dice+CE + original custom penalties" if IS_CUSTOM_LOSS else "Default nnU-Net Dice+CE only",
    "weights": {
        "missed_area": W_MISSED_AREA,
        "centroid_alignment": W_ALIGNMENT,
        "orientation": W_ORIENTATION,
        "alignment_grid": ALIGN_GRID_MAX_SIZE,
    } if IS_CUSTOM_LOSS else None,
    "local_model_output": str(model_output),
    "drive_model_output": str(canonical_model_dir),
    "saved_checkpoints": [p.name for p in checkpoint_files],
    "torch_version": torch.__version__,
}
(info_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2))

# Verify the *saved* final checkpoint contains the correct trainer / 51 epochs.
saved_final = legacy_model_dir / "checkpoint_final.pth"
saved = torch.load(saved_final, map_location="cpu", weights_only=False)
assert saved.get("trainer_name") == TRAINER_NAME
assert saved.get("current_epoch") == 51
del saved

print("\n=== SUCCESS: SAVED MODEL VERIFIED ===")
print("Trainer:", TRAINER_NAME)
print("Epochs: 51 | Cases:", len(preprocessed_case_ids))
print("Train / validation:", len(train_ids), "/", len(validation_ids))
print("Final checkpoint:", saved_final)
print("Best checkpoint:", legacy_model_dir / "checkpoint_best.pth" if (legacy_model_dir / "checkpoint_best.pth").is_file() else "not produced")
print("Canonical nnU-Net folder:", canonical_model_dir)
print("Split and manifest:", info_dir)

## 9. Results and comparison checklist

**This notebook performs training**, rather than loading an old checkpoint.

The saved `checkpoint_final.pth` is the trained model at epoch **51**. `checkpoint_best.pth` is the same model's best **validation** snapshot (if produced). **Neither is a pretrained or baseline checkpoint** in the custom notebook.

To run a fair **baseline-versus-custom** comparison:

1. Train each of the two notebooks with the same `Dataset901_PETCT_Baseline` cached dataset, `FOLD=0`, `EPOCHS=51`, seed `42`, plans and GPU configuration. Their Drive output roots are different.
2. Confirm **100** case IDs and matching `train_ids`/`validation_ids` and split checksum in `run_manifest.json`.
3. Evaluate both saved checkpoints with the **same full-volume validation cases**, identical TTA settings, and *the exact metadata saved with each model*. The original 9-case experiment's split must not be confused with this 100-case split.
4. Report Dice, IoU, precision, lesion recall, false-positive volume and missed lesion voxels. **Also include lesion-negative cases** for false-positive evaluation.

**Safety:** Do not point evaluation at an old `baseline_results/trained_model` folder. Those files were confirmed to identify themselves as `nnUNetTrainerCoverageAlign` (custom loss). Verify `trainer_name` inside each new checkpoint.

**Note:** `EPOCHS=51` means 51 nnU-Net epochs; it is not a 3-epoch warm start. The new notebook uses a new empty result directory to train from scratch.